# Street images: sample points, imagery by year, and the 10-image pilot

Read from `Data/cv_pilot_2026-10/` (made by `Scripts/cv_pilot/pilot.py`); nothing is recomputed or written.

1. The sample points in the CBD
2. How recent the street-level imagery is at each point
3. The 10 pilot images
4. What CLIP ViT-B/32 said about them

Nothing from CLIP is used until it has been checked against the ~150 hand-labelled images
(`Docs/validation_plan.md`).

In [ ]:
# Shared folders, project modules and plot settings
from _setup import *
# Show plots inside the notebook
%matplotlib inline

## Step 1: The sample points

Points about 50 m apart on OpenStreetMap streets, inside the CBD polygon.

In [ ]:
# One row per sample point
points = pd.read_csv(CV / "cbd" / "sample_points.csv")
print(f"{len(points)} points on {points['segment_id'].nunique()} street segments")
# Points per road class
points["highway"].value_counts().to_frame("points")

## Step 2: How recent the imagery is

For each point, the newest Mapillary image (flat or 360°) within 30 m.

In [ ]:
# One row per point: its newest flat and newest 360° image dates
coverage = pd.read_csv(CV / "cbd" / "coverage_points.csv")
# The newest capture year at each point, from either kind of image
newest = []
# Go through the points one at a time
for i in range(len(coverage)):
    # The years of the two newest dates that exist at this point
    years = [str(d)[:4] for d in (coverage["latest_flat"].iloc[i], coverage["latest_pano"].iloc[i]) if pd.notna(d)]
    # The later of the two, or "none" if the point has no image at all
    if years:
        newest.append(max(years))
    else:
        newest.append("none")
# Points per newest year
pd.Series(newest).value_counts().sort_index().to_frame("points")

## Step 3: The 10 pilot images

Chosen in a fixed spread order before any image was seen; the newest image within 30 m of each point. 360° images were cut into a side-facing view.

In [ ]:
# The 10 images and what CLIP scored for each
manifest = pd.read_csv(CV / "cbd_latest_imagery" / "image_manifest.csv")
scores = pd.read_csv(CV / "cbd_latest_imagery" / "clip_scores.csv")
pilot = manifest.merge(scores[["point_id", "typology_top1", "typology_top1_prob"]], on="point_id")
# Two rows of five images
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))
# Draw each image with its point, capture year and CLIP's top land-use label
for i in range(len(pilot)):
    # The panel for this image
    ax = axes.flat[i]
    ax.imshow(plt.imread(CV / "cbd_latest_imagery" / pilot["image_file"].iloc[i]))
    ax.set_title(f"{pilot['point_id'].iloc[i]} ({pilot['capture_year'].iloc[i]})\n"
                 f"{pilot['typology_top1'].iloc[i]} {pilot['typology_top1_prob'].iloc[i]:.2f}", fontsize=8)
    ax.axis("off")
# Hide any panel left empty
for j in range(len(pilot), len(axes.flat)):
    axes.flat[j].axis("off")
fig.text(0.01, 0.01, MAPILLARY_CREDIT, fontsize=8)
plt.tight_layout()
plt.show()

## Step 4: CLIP's land-use labels

The top label and its probability; a small margin over the second label marks a low-confidence call.

In [ ]:
# CLIP's top two land-use labels for each image, and whether the call is low-confidence
scores[["point_id", "typology_top1", "typology_top1_prob", "typology_top2", "typology_top2_prob", "typology_low_confidence"]]

## Takeaway

- No image from June–August 2026 exists at any CBD point; the newest anywhere in the CBD is from 2023
  (Step 2 gives each point's newest year; Decision D1 in `Docs/cv_pipeline.md`).
- CLIP runs on the pilot images, but its labels are not measurements until they agree with hand labels.